<div style="
background-color:#EAEAEA;
padding:20px;
border-left:5px solid #6C757D;
border-radius:6px;">

<table style="width:100%; border:none;">
<tr style="border:none;">

<td style="border:none; vertical-align:top;">

<h1 style="font-size:32px; margin-top:0;">
Master's Thesis
</h1>

<hr style="margin:16px 0 22px 0;">

<p style="font-size:22px; line-height:1.5; margin:0;">
<strong>Master's Degree in Advanced Physics</strong> - 
<strong>Universitat de Val?ncia</strong>
</p>

<p style="font-size:17px; margin-top:28px; margin-bottom:6px;">
This notebook is part of the <strong>Master's Thesis (MSc Dissertation)</strong>:
</p>

<div style="
font-size:25px;
font-weight:700;
line-height:1.3;
margin-top:14px;
margin-bottom:26px;">
Fast Simulation of Neutrino Oscillations in Matter
</div>

<p style="font-size:14px; line-height:1.55;">
<strong>Author</strong><br>
Juan Ramon Diaz Santos - 
<a href="mailto:diazjuan@alumni.uv.es">diazjuan@alumni.uv.es</a>
</p>

<p style="font-size:14px; line-height:1.55;">
<strong>Supervisors</strong><br>
Roberto Ruiz de Austri Bazan ?
<a href="mailto:rruiz@ific.uv.es">rruiz@ific.uv.es</a><br>
Michele Lucente ?
<a href="mailto:michele.lucente@unibo.it">michele.lucente@unibo.it</a>
</p>

<p style="font-size:14px; line-height:1.55; margin-bottom:0;">
<strong>Date</strong><br>
September 2026
</p>

</td>

<td style="
border:none;
width:230px;
padding-left:25px;
text-align:right;
vertical-align:top;">

<img src="../../logo_uv.png"
     alt="Universitat de Val?ncia"
     style="width:200px; margin-top:5px;">

</td>

</tr>
</table>

</div>

# Atmosphere 6 — Detector Flux Generation

Propagates height-differential atmospheric neutrino fluxes $\Phi(E,\theta,h)$ through
the atmosphere and Earth to produce the observable detector-level flux
$\Phi_{\mathrm{det}}(E,\theta)$.

Input data can come from either the Honda-table generator (Atmosphere 3) or the
MCEq cascade solver (Atmosphere 5). The full propagation chain is:

$$\Phi(E,\theta,h)
  \xrightarrow{\text{atmosphere}} \Phi_{\mathrm{surf}}(E,\theta,h)
  \xrightarrow{\text{Earth}}     \Phi_{\mathrm{det}}(E,\theta,h)
  \xrightarrow{\int dh}          \Phi_{\mathrm{det}}(E,\theta)$$

---

## Table of Contents

| # | Section |
|---|---------|
| [0](#0.-Theory-Background) | **Theory Background**: atmospheric production, coherent propagation, Earth effects, height integration |
| [1](#1.-Libraries) | **Libraries** |
| [2](#2.-Configuration) | **Configuration** — paths, parameters, helpers |
| [3](#3.-Load-Differential-Flux) | **Load Differential Flux**: input data inspection |
| [4](#4.-Single-Angle-Propagation) | **Single-Angle Propagation**: diagnostic chain for one particle/angle |
| [5](#5.-Full-Grid-Propagation) | **Full Grid Propagation**: save all particle/angle pairs |
| [6](#6.-Detector-Flux-Analysis) | **Detector Flux Analysis**: aggregate and visualize |
| [&#x2211;](#7.-Summary) | **Summary** |

## 0. Theory Background

### 0.1 Atmospheric Neutrino Production

Cosmic rays striking the upper atmosphere initiate hadronic cascades that produce
pions ($\pi^\pm$) and kaons ($K^\pm$). Their decays generate the atmospheric
neutrino beam:

$$\pi^+ \to \mu^+ + \nu_\mu, \qquad \mu^+ \to e^+ + \nu_e + \bar\nu_\mu,$$

and the charge-conjugate processes for antineutrinos. The neutrino source is
therefore height-distributed: the **production altitude** $h$ depends on the
parent meson's Lorentz factor (higher $E_\nu$ implies higher $h$ at fixed $\theta$)
and on the atmospheric density profile. For a given zenith angle $\theta$ and energy
$E_\nu$, the source is characterised by the height-differential flux

$$\Phi(E,\theta,h) \;=\; \Phi(E,\theta) \,\times\, f(h\,|\,E,\theta),$$

where $f(h|E,\theta)$ is the normalised production-height PDF and $\Phi(E,\theta)$
is the height-integrated flux. This factorisation is what Honda tables and MCEq both
provide (see Atmospheres 2–5).

---

### 0.2 Coherent Propagation Through the Atmosphere

Each produced neutrino travels from height $h$ to the Earth's surface along a slant
distance

$$L_{\mathrm{atm}}(h,\theta)
  = \sqrt{(R_\oplus + h)^2 - R_\oplus^2\sin^2\theta} - R_\oplus\cos\theta,$$

where $R_\oplus = 6371$ km. Along this path the flavour state evolves coherently under
the full three-flavour Hamiltonian $H = H_{\mathrm{vac}} + V_{\mathrm{atm}}$, where
the atmospheric matter potential $V_{\mathrm{atm}} = \sqrt{2}\,G_F\,n_e^{\mathrm{atm}}$
uses the NRLMSISE-00 density profile *(Honda et al. 2015)*. The surface state after
traversing height $h$ is

$$|\psi_{\mathrm{surf}}(E,h,\theta)\rangle
  = U_{\mathrm{atm}}(E,h,\theta)\,|\nu_\beta\rangle,$$

where $|\nu_\beta\rangle$ is the produced flavour eigenstate and $U_{\mathrm{atm}}$
is the atmosphere evolution operator. The atmospheric matter effect is small
($n_e^{\mathrm{atm}} \ll n_e^{\mathrm{Earth}}$) but is retained for physical
completeness.

---

### 0.3 Earth Matter Effects and Coherent Propagation

For **up-going neutrinos** ($\theta > 90°$) the surface state continues through
the Earth's interior. The Earth matter potential follows the electron-density profile
from seismic data (PREM-based), modelled in tpeanuts as an even-power polynomial fit.
The Earth evolution operator $U_{\mathrm{Earth}}(E,\theta)$ is independent of $h$
(the Earth path is fixed by detector geometry), so it is computed once per $(E,\theta)$
pair and applied to all $h$-values:

$$|\psi_{\mathrm{det}}(E,h,\theta)\rangle
  = U_{\mathrm{Earth}}(E,\theta)\,|\psi_{\mathrm{surf}}(E,h,\theta)\rangle.$$

The detector-level oscillation probability is

$$P(\nu_\beta \to \nu_i;\, E,\theta,h)
  = |\langle\nu_i|\psi_{\mathrm{det}}(E,h,\theta)\rangle|^2.$$

For down-going neutrinos ($\theta < 90°$) the Earth path length vanishes and
$U_{\mathrm{Earth}} = \mathbf{1}$.

---

### 0.4 Height Integration — Incoherent in Flux

Neutrinos produced at different heights $h$ are physically distinguishable (different
path lengths $\Rightarrow$ different oscillation phases) and therefore combine
**incoherently in flux** *(Barger et al. 2012)*:

$$\Phi_{\mathrm{det}}^{\beta \to i}(E,\theta)
  = \int_0^{h_{\mathrm{top}}} dh\;\Phi(E,\theta,h)\,P(\nu_\beta \to \nu_i;\,E,\theta,h).$$

The effective height-averaged detector probability is then

$$P_{\mathrm{det}}^{\beta \to i}(E,\theta)
  = \frac{\Phi_{\mathrm{det}}^{\beta \to i}(E,\theta)}%
         {\displaystyle\int dh\;\Phi(E,\theta,h)}.$$

This is the same height-averaging prescription established for atmospheric oscillation
analyses at Super-Kamiokande *(Fukuda et al. 1998)*.

---

### 0.5 The Three-Flavour Detector Flux

The total detector flux in each final flavour $i \in \{\nu_e,\nu_\mu,\nu_\tau\}$
is the sum over all produced flavours $\beta$:

$$\Phi_{\mathrm{det}}^{i}(E,\theta) = \sum_\beta \Phi_{\mathrm{det}}^{\beta \to i}(E,\theta).$$

For atmospheric neutrinos the produced flavours are
$\{\nu_e,\nu_\mu,\bar\nu_e,\bar\nu_\mu\}$. Because the antineutrino matter potential
has the opposite sign ($V \to -V$), neutrinos and antineutrinos are propagated
separately with the appropriate `antinu` flag.

---

**References**
- Wolfenstein, L. (1978). *Neutrino oscillations in matter*. Phys. Rev. D **17**, 2369.
- Fukuda, Y. et al. (Super-Kamiokande Collaboration, 1998).
  *Evidence for an anomalous disappearance of atmospheric $\nu_\mu$*.
  Phys. Rev. Lett. **81**, 1562.
- Barger, V., Marfatia, D. & Whisnant, K. (2012). *The Physics of Neutrinos*,
  Chapter 8. Princeton University Press.
- Giunti, C. & Kim, C. W. (2007). *Fundamentals of Neutrino Physics and Astrophysics*,
  §12–13. Oxford University Press.
- Honda, M. et al. (2015). *Atmospheric neutrino flux calculation using the
  NRLMSISE-00 atmospheric model*. Phys. Rev. D **92**, 023004.

## 1. Libraries

All imports for the notebook. Physics pipeline entry points are in `tpeanuts.pipeline`;
`load_directory` reloads the stacked production flux. Plotting uses matplotlib with the
shared style from `notebookConfig`.

In [7]:
from __future__ import annotations

import time
from dataclasses import replace
from pathlib import Path
from typing import Dict, Optional

import numpy as np
import torch
import matplotlib.pyplot as plt

from tpeanuts.notebooks.notebookConfig import load_notebook_config
from tpeanuts.notebooks.notebooks_helper import save_and_show

from tpeanuts.core.common.oscillation import OscillationParameters
from tpeanuts.config.propagation import PropagationConfig
from tpeanuts.medium.atmosphere.io import load_directory
from tpeanuts.medium.atmosphere.profile import AtmosphereParameters
from tpeanuts.medium.earth.profile import EarthParameters, EarthProfile
from tpeanuts.config.propagation import PropagationConfig
from tpeanuts.pipeline import (
    propagate_atmosphere_to_surface,
    propagate_surface_to_detector,
    select_production_flux,
    aggregate_detector_flux_by_mode,
    build_detector_flux_path,
    load_detector_flux_directory,
    save_detector_flux_result,
)
from tpeanuts.pipeline.atmosphere_earth import (
    detector_flux_from_production, integrate_detector_flux_over_height,
    sum_detected_flavours,
)
from tpeanuts.util.context import RuntimeContext
from tpeanuts.util.torch_util import default_device, resolve_device
from tpeanuts.util.type import as_tensor, to_numpy

print(f"torch : {torch.__version__}")
print(f"numpy : {np.__version__}")

torch : 2.5.1+cu121
numpy : 1.26.2


## 2. Configuration

### 2.1 Paths

`load_notebook_config()` resolves the package root and output root, and applies the
shared matplotlib / torch style. Differential flux data can come from either Honda
(Atmosphere 3) or MCEq (Atmosphere 5) — swap `INPUT_DIR` to select the source.
Detector flux outputs are written under `data/flux/detector/runXXXX`, one file per
(particle, angle).

In [8]:
config      = load_notebook_config()
PACKAGE_DIR = config.package_dir
OUTPUT_ROOT = config.output_root
OUTPUT_DIR  = config.output_dir("analysis", "atmosphere")

EARTH_DENSITY_FILE = str(PACKAGE_DIR / "data" / "earth" / "prem" / "fit" / "even_power_electron.csv")

# --- Input: differential flux directory (Honda or MCEq) ---
# MCEq (Atmosphere 5 output):
INPUT_DIR = str(
    OUTPUT_ROOT / "data" / "flux" / "differential" / "honda"
    / "run0001"
)
# Honda (Atmosphere 3 output) — uncomment to use instead:
# INPUT_DIR = str(
#     OUTPUT_ROOT / "data" / "atmosphere" / "honda"
#     / "diff_flux_frj_ally_solmin"
# )

# --- Output: detector flux (auto-incremented runXXXX) ---
DETECTOR_BASE_DIR = OUTPUT_ROOT / "data" / "flux" / "detector"
DETECTOR_BASE_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILENAME = "detector_flux.pt"

print(f"Package dir    : {PACKAGE_DIR}")
print(f"Output dir     : {OUTPUT_DIR}")
print(f"Input flux dir : {INPUT_DIR}")
print(f"Detector base  : {DETECTOR_BASE_DIR}")
print(f"Earth density  : {EARTH_DENSITY_FILE}")

Package dir    : G:\Mi unidad\03.Codigo\034.TFM.UV\Tpeanuts
Output dir     : v:\output\analysis\atmosphere
Input flux dir : v:\output\data\flux\differential\honda\run0001
Detector base  : v:\output\data\flux\detector
Earth density  : G:\Mi unidad\03.Codigo\034.TFM.UV\Tpeanuts\data\earth\prem\density\prem_density.csv


### 2.2 Configuration

Physical and numerical parameters for the propagation run. The oscillation benchmark
is NuFIT 5.2 Normal Ordering. `ENERGY_CHUNK_SIZE` limits memory per evolutor call;
`ATMOSPHERE_STEPS` controls numerical path-integration accuracy.

| Parameter | Value | Description |
|-----------|-------|-------------|
| Oscillation preset | `_SM_NUFIT52_NO` | NuFIT 5.2, Normal Ordering |
| `DETECTOR_DEPTH_M` | 1000 m | Detector depth below surface |
| `MATTER_IN_ATMOSPHERE` | `True` | Include atmospheric matter potential |
| `REUNITARIZE_EARTH` | `False` | Project Earth operator onto $U(3)$ |
| `ATMOSPHERE_STEPS` | 120 | Integration steps through atmosphere |
| `TRAJECTORY_STEPS` | 120 | Diagnostic trajectory grid points |
| `ENERGY_CHUNK_SIZE` | 8 | Energies per batched evolutor call |
| `SAVE_DTYPE` | `torch.float32` | On-disk precision ($\sim 10^{-7}$ relative) |
| `COMPUTE_DTYPE` | `torch.float64` | Internal compute precision |
| `SKIP_EXISTING` | `True` | Skip already-saved particle/angle files |
| `OVERWRITE` | `False` | Overwrite existing output files |
| `HEIGHT_MAX_COUNT` | `None` | Subsample height grid (`None` = full) |
| `ENERGY_MAX_COUNT` | `None` | Subsample energy grid (`None` = full) |
| `ANGLE_MAX_COUNT` | `None` | Subsample angle grid (`None` = full) |

In [9]:
LOAD_DEVICE        = "cpu"
PROPAGATION_DEVICE = default_device
COMPUTE_DTYPE      = torch.float64
LOAD_DTYPE         = torch.float64
SAVE_DTYPE         = torch.float32

ENERGY_CHUNK_SIZE: Optional[int] = 8
HEIGHT_MAX_COUNT:  Optional[int] = None
ENERGY_MAX_COUNT:  Optional[int] = None
ANGLE_MAX_COUNT:   Optional[int] = None

ATMOSPHERE_STEPS = 120
TRAJECTORY_STEPS = 120

DETECTOR_DEPTH_M     = 1000.0
MATTER_IN_ATMOSPHERE = True
REUNITARIZE_EARTH    = False

SKIP_EXISTING = True
OVERWRITE     = False
DEBUG         = True

PARTICLES: Optional[tuple[str, ...]] = None  # None = use all available particles

# Display oscillation parameters
_ctx_osc = RuntimeContext.resolve("cpu", COMPUTE_DTYPE)
_osc     = PropagationConfig.oscillation_parameters_from_preset(context=_ctx_osc, antinu=False)
_pmns    = _osc.pmns

THETA12  = float(_pmns.params.theta12.detach().cpu())
THETA13  = float(_pmns.params.theta13.detach().cpu())
THETA23  = float(_pmns.params.theta23.detach().cpu())
DELTA_CP = float(_pmns.params.delta.detach().cpu())
DM21_EV2 = float(_osc.mass_spectrum.DeltamSq21.detach().cpu())
DM3L_EV2 = float(_osc.mass_spectrum.DeltamSq3l.detach().cpu())

print(f"Oscillation preset  : {_osc.preset_name} ({_osc.ordering})")
print(f"theta12 / theta13   : {THETA12*180/np.pi:.2f} / {THETA13*180/np.pi:.2f} deg")
print(f"theta23             : {THETA23*180/np.pi:.2f} deg")
print(f"delta_CP            : {DELTA_CP*180/np.pi:.2f} deg")
print(f"DeltamSq21          : {DM21_EV2:.4e} eV2")
print(f"DeltamSq3l          : {DM3L_EV2:.4e} eV2")
print(f"Detector depth      : {DETECTOR_DEPTH_M:.0f} m")
print(f"Matter in atm       : {MATTER_IN_ATMOSPHERE}")

Oscillation preset  : _SM_NUFIT52_NO (NO)
theta12 / theta13   : 33.41 / 8.58 deg
theta23             : 49.00 deg
delta_CP            : 197.00 deg
DeltamSq21          : 7.4100e-05 eV2
DeltamSq3l          : 2.5110e-03 eV2
Detector depth      : 1000 m
Matter in atm       : True


### 2.3 Helpers

Minimal set of helpers required by the propagation loop. All but `next_run_dir` mirror
the implementation in `atmosphere2_detector_flux_propagation` (the production run
notebook).

- `next_run_dir` — auto-increments the `runXXXX` output directory.
- `evenly_spaced_indices` — subsamples a 1-D grid to a maximum count.
- `chunk_indices` — splits an index tensor into chunks of `chunk_size`.
- `infer_antinu` — detects antineutrino particles from the particle key name.
- `thin_selected` — subsets a `select_production_flux` dict onto requested
  energy/height index subsets.
- `propagate_particle_angle` — runs the full atmosphere → Earth → height-integration
  chain for one (particle, angle) pair in energy chunks.

In [10]:
def next_run_dir(base_dir: Path) -> Path:
    existing = sorted(
        int(p.name[3:])
        for p in base_dir.iterdir()
        if p.is_dir() and p.name.startswith("run") and p.name[3:].isdigit()
    ) if base_dir.exists() else []
    idx = (existing[-1] + 1) if existing else 1
    run_dir = base_dir / f"run{idx:04d}"
    run_dir.mkdir(parents=True, exist_ok=True)
    return run_dir


def evenly_spaced_indices(n_items: int, max_count: Optional[int] = None) -> torch.Tensor:
    if max_count is None or max_count >= n_items:
        return torch.arange(n_items, dtype=torch.long)
    return (
        torch.linspace(0, n_items - 1, int(max_count), dtype=torch.float64)
        .round()
        .to(torch.long)
        .unique()
    )


def chunk_indices(indices: torch.Tensor, chunk_size: Optional[int]):
    indices = indices.reshape(-1).to(dtype=torch.long)
    if chunk_size is None or chunk_size <= 0:
        yield indices
        return
    for start in range(0, indices.numel(), int(chunk_size)):
        yield indices[start : start + int(chunk_size)]


def infer_antinu(particle: str) -> bool:
    key = str(particle).lower()
    return "anti" in key or "bar" in key


def thin_selected(
    selected: Dict[str, object],
    *,
    energy_indices: torch.Tensor,
    height_indices: torch.Tensor,
) -> Dict[str, object]:
    thinned  = dict(selected)
    E        = selected["E_grid_GeV"][energy_indices]
    h        = selected["h_grid_km"][height_indices]
    phi_Eh   = selected["phi_Eh"][energy_indices][:, height_indices]
    thinned["E_grid_GeV"]  = E
    thinned["h_grid_km"]   = h
    thinned["phi_Eh"]      = phi_Eh
    thinned["phi_E_theta"] = torch.trapezoid(phi_Eh, x=h, dim=-1)
    if selected.get("f_Eh", None) is not None:
        thinned["f_Eh"] = selected["f_Eh"][energy_indices][:, height_indices]
    return thinned


def propagate_particle_angle(
    *,
    selected: Dict[str, object],
    config: PropagationConfig,
    profile_earth: EarthProfile,
    energy_indices: torch.Tensor,
    height_indices: torch.Tensor,
) -> Dict[str, torch.Tensor]:
    context = config.runtime
    det_flux, surf_flux, init_flux = [], [], []
    det_prob, surf_prob, src_flux, E_list = [], [], [], []

    for chunk in chunk_indices(energy_indices, ENERGY_CHUNK_SIZE):
        sel_c = thin_selected(
            selected,
            energy_indices=chunk,
            height_indices=height_indices,
        )
        atm = propagate_atmosphere_to_surface(
            sel_c, config, trajectory_steps=TRAJECTORY_STEPS
        )
        ear = propagate_surface_to_detector(
            atm, config, profile_earth=profile_earth
        )

        p   = sel_c["particle"]
        phi = sel_c["phi_Eh"]
        h = sel_c["h_grid_km"]
        source = torch.trapezoid(phi, x=h, dim=-1)
        inf = torch.zeros((*source.shape, 3), device=context.device, dtype=context.dtype)
        inf[..., sel_c["flavour_index"]] = source
        suf = integrate_detector_flux_over_height(h, detector_flux_from_production(phi, atm.surface_probabilities))
        df = integrate_detector_flux_over_height(h, detector_flux_from_production(phi, ear.detector_probabilities))
        sup = suf / source[..., None].clamp_min(torch.finfo(context.dtype).tiny)
        sf  = torch.sum(inf, dim=-1)
        dp  = df / sf[..., None].clamp_min(torch.finfo(context.dtype).tiny)

        E_list.append(sel_c["E_grid_GeV"])
        det_flux.append(df)
        surf_flux.append(suf)
        init_flux.append(inf)
        det_prob.append(dp)
        surf_prob.append(sup)
        src_flux.append(sf)

    return {
        "E_grid_GeV":              torch.cat(E_list,    dim=0),
        "detector_flux_Ei":        torch.cat(det_flux,  dim=0),
        "surface_flux_Ei":         torch.cat(surf_flux, dim=0),
        "initial_flux_Ei":         torch.cat(init_flux, dim=0),
        "detector_probability_Ei": torch.cat(det_prob,  dim=0),
        "surface_probability_Ei":  torch.cat(surf_prob, dim=0),
        "source_flux_E":           torch.cat(src_flux,  dim=0),
    }

## 3. Load Differential Flux

Loads the stacked height-differential production flux $\Phi(E,\theta,h)$ from
`INPUT_DIR`. The `load_directory` function groups files by particle and assembles
the tensors `phi_E_theta_h` (shape `(n_angle, n_E, n_h)`), `E_grid_GeV`,
`h_grid_km`, and `theta_grid_deg`.

Both Honda (Atmosphere 3) and MCEq (Atmosphere 5) outputs expose the same data
contract, so the propagation code works identically with either source.

**Expected results:** A summary of available particles, grid sizes, and zenith angle
coverage. Typical MCEq output: 4 particles, ~101 energies, ~501 height points,
~10 zenith angles.

In [11]:
load_device = resolve_device(LOAD_DEVICE)

flux_data = load_directory(
    INPUT_DIR,
    map_location="cpu",
    dtype=LOAD_DTYPE,
    device=load_device,
    group_by="particle",
    verbose=DEBUG,
)

particles = list(PARTICLES) if PARTICLES is not None else sorted(flux_data.keys())
print(f"\nAvailable particles : {sorted(flux_data.keys())}")
print(f"Selected particles  : {particles}")
print()

for ptcl in particles:
    g = flux_data[ptcl]
    n_ang = int(g["theta_grid_deg"].numel())
    n_E   = int(g["E_grid_GeV"].numel())
    n_h   = int(g["h_grid_km"].numel())
    th_lo = float(g["theta_grid_deg"].min())
    th_hi = float(g["theta_grid_deg"].max())
    E_lo  = float(g["E_grid_GeV"].min())
    E_hi  = float(g["E_grid_GeV"].max())
    print(
        f"  {ptcl:14s}: n_ang={n_ang:3d}, n_E={n_E:4d}, n_h={n_h:4d}"
        f"  theta=[{th_lo:.1f},{th_hi:.1f}] deg"
        f"  E=[{E_lo:.3g},{E_hi:.3g}] GeV"
    )

Loaded antinue      | n_theta =  10 | phi(E,theta,h) shape = (10, 101, 501)
Loaded antinumu     | n_theta =  10 | phi(E,theta,h) shape = (10, 101, 501)
Loaded antinutau    | n_theta =  10 | phi(E,theta,h) shape = (10, 101, 501)
Loaded nue          | n_theta =  10 | phi(E,theta,h) shape = (10, 101, 501)
Loaded numu         | n_theta =  10 | phi(E,theta,h) shape = (10, 101, 501)
Loaded nutau        | n_theta =  10 | phi(E,theta,h) shape = (10, 101, 501)

Available particles : ['antinue', 'antinumu', 'antinutau', 'nue', 'numu', 'nutau']
Selected particles  : ['antinue', 'antinumu', 'antinutau', 'nue', 'numu', 'nutau']

  antinue       : n_ang= 10, n_E= 101, n_h= 501  theta=[18.2,87.1] deg  E=[0.1,1e+04] GeV
  antinumu      : n_ang= 10, n_E= 101, n_h= 501  theta=[18.2,87.1] deg  E=[0.1,1e+04] GeV
  antinutau     : n_ang= 10, n_E= 101, n_h= 501  theta=[18.2,87.1] deg  E=[0.1,1e+04] GeV
  nue           : n_ang= 10, n_E= 101, n_h= 501  theta=[18.2,87.1] deg  E=[0.1,1e+04] GeV
  numu          

## 4. Single-Angle Propagation

Before running the full grid we propagate a single (particle, angle) pair to inspect
intermediate quantities:

- **Surface oscillation probabilities** $P_{\rm surf}(E,h)$ — after atmosphere
  crossing.
- **Detector flux** $\Phi_{\rm det}(E)$ — after Earth crossing and height
  integration.

This diagnostic verifies that the oscillation physics is correct before committing
to the full production run.

**Expected results:**
- For down-going trajectories ($\theta < 90°$), the atmosphere path is short
  ($\lesssim 20$ km slant) and the matter effect is negligible; the surface
  probabilities are close to the vacuum values at the corresponding $L/E$.
- For up-going trajectories, Earth crossing introduces strong $\nu_\mu$
  disappearance at GeV energies via the atmospheric mass splitting
  $\Delta m^2_{3l}$.
- The height-integrated detector flux preserves the steep power-law shape of
  the source but is redistributed among flavours by oscillations.

In [12]:
DIAG_PARTICLE  = particles[1] if len(particles) > 1 else particles[0]
DIAG_ANGLE_IDX = 0    # index into the loaded angle grid; change to explore

propagation_device = resolve_device(PROPAGATION_DEVICE)
context = RuntimeContext(device=propagation_device, dtype=COMPUTE_DTYPE)
oscillation = PropagationConfig.oscillation_parameters_from_preset(context=context, antinu=False)

profile_earth = EarthProfile(
    params=EarthParameters(
        profile_perturbative_name="even_power",
        profile_perturbative_kwargs={"density_file": EARTH_DENSITY_FILE},
    ),
    context=context,
)

base_config = PropagationConfig(
    runtime=context,
    oscillation=oscillation,
    earth=EarthParameters(
        profile_perturbative_name="even_power",
        profile_perturbative_kwargs={"density_file": EARTH_DENSITY_FILE},
    ),
    atmosphere=AtmosphereParameters(
        nsteps=ATMOSPHERE_STEPS,
        matter=MATTER_IN_ATMOSPHERE,
    ),
    detector_depth_m=DETECTOR_DEPTH_M,
    reunitarize_earth=REUNITARIZE_EARTH,
)

diag_selected = select_production_flux(
    flux_data,
    DIAG_PARTICLE,
    angle_index=DIAG_ANGLE_IDX,
    context=context,
)
theta_diag = float(diag_selected["theta_deg"])

FLAVOUR_LABELS = [r"$\nu_e$", r"$\nu_\mu$", r"$\nu_\tau$"]   # for legend labels
FLAVOUR_TEX    = [r"\nu_e",   r"\nu_\mu",   r"\nu_\tau"]      # for inside $...$
FLAVOUR_COLORS = ["#dd8452", "#4c72b0", "#55a868"]

print(f"Diagnostic particle : {DIAG_PARTICLE}")
print(f"Zenith angle        : {theta_diag:.2f} deg")
print(
    f"Energy range        : "
    f"{float(diag_selected['E_grid_GeV'][0]):.3g}"
    f" -- {float(diag_selected['E_grid_GeV'][-1]):.3g} GeV"
)
print(
    f"Height range        : "
    f"{float(diag_selected['h_grid_km'][0]):.1f}"
    f" -- {float(diag_selected['h_grid_km'][-1]):.1f} km"
)

KeyError: 'rj'

### 4.1 Surface Probabilities after Atmosphere Crossing

The heatmap shows $P(\nu_\beta \to \nu_i;\, E, h)$ at the Earth's surface as a
function of energy and production height, for the selected particle and angle. Each
row is one energy; each column is one production height; the colour encodes the
probability of each final flavour.

**Expected results:** For short down-going paths, $P(\nu_\mu \to \nu_\mu)$ remains
close to 1 across most of the $(E, h)$ grid. Significant conversion appears only for
large $h$ (long slant paths) combined with low energies where the atmospheric phase
$\Delta m^2 L / 4E$ is non-negligible.

In [ ]:
diag_config = replace(
    base_config,
    oscillation=replace(oscillation, antinu=infer_antinu(DIAG_PARTICLE)),
)

e_idx = evenly_spaced_indices(int(diag_selected["E_grid_GeV"].numel()), ENERGY_MAX_COUNT)
h_idx = evenly_spaced_indices(int(diag_selected["h_grid_km"].numel()),  HEIGHT_MAX_COUNT)
diag_thin = thin_selected(diag_selected, energy_indices=e_idx, height_indices=h_idx)

atm_result = propagate_atmosphere_to_surface(
    diag_thin, diag_config, trajectory_steps=TRAJECTORY_STEPS
)

surf_prob = to_numpy(atm_result.surface_probabilities)  # (n_E, n_h, 3)
E_diag    = to_numpy(diag_thin["E_grid_GeV"])
h_diag    = to_numpy(diag_thin["h_grid_km"])

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for fi in range(3):
    ax = axes[fi]
    vmax = max(float(surf_prob[..., fi].max()), 1e-6)
    im = ax.pcolormesh(
        h_diag, E_diag, surf_prob[..., fi],
        cmap="YlOrRd", shading="auto", vmin=0, vmax=vmax,
    )
    ax.set_yscale("log")
    ax.set_xlabel("Production height $h$ [km]")
    ax.set_ylabel("$E$ [GeV]" if fi == 0 else "")
    ax.set_title(rf"$P(\nu \to {FLAVOUR_TEX[fi]})$ at surface", fontsize=9)
    fig.colorbar(im, ax=ax, label="Probability")
    ax.grid(True, which="both", alpha=0.3)

fig.suptitle(
    rf"Surface oscillation probabilities — {DIAG_PARTICLE}, $\theta={theta_diag:.1f}^\circ$",
    fontsize=10,
)
fig.tight_layout()
save_and_show("atm6_fig41_surface_prob.png", fig, output_dir=OUTPUT_DIR, show_plots=config.show_plots)

### 4.2 Detector Flux after Full Propagation

The height-integrated detector flux $\Phi_{\mathrm{det}}(E)$ for each final flavour
after the complete atmosphere → Earth → height-integration chain. The left panel shows
absolute fluxes; the right panel shows the effective oscillation probability
$P_{\rm det}^{\beta \to i}(E) = \Phi_{\rm det}^i(E) / \Phi_{\rm src}(E)$.

**Expected results:** The dominant channel is survival ($\nu_\beta \to \nu_\beta$).
For down-going trajectories the spectrum keeps the power-law shape of the source;
for up-going paths the muon-neutrino channel shows a characteristic dip at
$\sim 1$–$10$ GeV driven by $\Delta m^2_{3l}$ oscillations.

In [ ]:
ear_result = propagate_surface_to_detector(
    atm_result, diag_config, profile_earth=profile_earth
)

det_flux_diag = to_numpy(integrate_detector_flux_over_height(
    diag_thin["h_grid_km"],
    detector_flux_from_production(diag_thin["phi_Eh"], ear_result.detector_probabilities),
))
source_diag = torch.trapezoid(diag_thin["phi_Eh"], x=diag_thin["h_grid_km"], dim=-1)
init_flux_diag = np.zeros((*to_numpy(source_diag).shape, 3))
init_flux_diag[..., diag_thin["flavour_index"]] = to_numpy(source_diag)
src_total     = init_flux_diag.sum(axis=-1)   # (n_E,)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
for fi, (flabel, fcol) in enumerate(zip(FLAVOUR_LABELS, FLAVOUR_COLORS)):
    ax.loglog(E_diag, det_flux_diag[:, fi], color=fcol, lw=2, label=f"det {flabel}")
ax.loglog(E_diag, src_total, "k--", lw=1.5, label="source (initial)")
ax.set_xlabel("$E$ [GeV]")
ax.set_ylabel(r"$\Phi$ [(cm$^2$ s sr GeV)$^{-1}$]")
ax.set_title("Detector flux by final flavour", fontsize=10)
ax.legend(fontsize=8)
ax.grid(True, which="both", alpha=0.3)

ax = axes[1]
eps = torch.finfo(torch.float64).tiny
for fi, fcol in enumerate(FLAVOUR_COLORS):
    ratio = np.where(src_total > eps, det_flux_diag[:, fi] / src_total, np.nan)
    ax.semilogx(E_diag, ratio, color=fcol, lw=2, label=rf"$P(\nu \to {FLAVOUR_TEX[fi]})$")
ax.set_xlabel("$E$ [GeV]")
ax.set_ylabel("Height-averaged probability")
ax.set_title("Effective oscillation probability", fontsize=10)
ax.legend(fontsize=8)
ax.grid(True, which="both", alpha=0.3)
ax.set_ylim(-0.05, 1.05)

fig.suptitle(
    rf"Detector flux — {DIAG_PARTICLE}, $\theta={theta_diag:.1f}^\circ$",
    fontsize=10,
)
fig.tight_layout()
save_and_show("atm6_fig42_detector_flux.png", fig, output_dir=OUTPUT_DIR, show_plots=config.show_plots)

## 5. Full Grid Propagation and Save

Runs the complete atmosphere → Earth → height-integration chain for every
(particle, angle) pair in the loaded flux data, saving one `.pt` file per pair
to an auto-incremented `runXXXX` subdirectory. Each file contains the production,
surface, and detector fluxes together with oscillation probabilities and run
metadata, making each file self-describing.

`SKIP_EXISTING=True` resumes an interrupted run without recomputing already-saved
pairs. Set `OVERWRITE=True, SKIP_EXISTING=False` to force regeneration.

**Expected results:** One output file per (particle, angle) combination, with
progress lines printed for each pair and elapsed time reported at the end.

In [ ]:
def run_propagation(output_run_dir: Path) -> list[str]:
    prop_dev = resolve_device(PROPAGATION_DEVICE)
    ctx      = RuntimeContext(device=prop_dev, dtype=COMPUTE_DTYPE)
    osc      = PropagationConfig.oscillation_parameters_from_preset(context=ctx, antinu=False)
    ep       = EarthProfile(
        params=EarthParameters(
            profile_perturbative_name="even_power",
            profile_perturbative_kwargs={"density_file": EARTH_DENSITY_FILE},
        ),
        context=ctx,
    )
    bcfg = PropagationConfig(
        runtime=ctx,
        oscillation=osc,
        earth=EarthParameters(
            profile_perturbative_name="even_power",
            profile_perturbative_kwargs={"density_file": EARTH_DENSITY_FILE},
        ),
        atmosphere=AtmosphereParameters(
            nsteps=ATMOSPHERE_STEPS,
            matter=MATTER_IN_ATMOSPHERE,
        ),
        detector_depth_m=DETECTOR_DEPTH_M,
        reunitarize_earth=REUNITARIZE_EARTH,
    )

    print(f"\nPropagation run")
    print(f"  Input dir    : {INPUT_DIR}")
    print(f"  Output dir   : {output_run_dir}")
    print(f"  Particles    : {particles}")

    saved = []
    t0 = time.perf_counter()

    for ptcl in particles:
        if ptcl not in flux_data:
            print(f"  Missing particle: {ptcl}")
            continue

        g       = flux_data[ptcl]
        n_ang   = int(g["theta_grid_deg"].numel())
        ang_idx = evenly_spaced_indices(n_ang, ANGLE_MAX_COUNT)
        e_idx   = evenly_spaced_indices(int(g["E_grid_GeV"].numel()), ENERGY_MAX_COUNT)
        h_idx   = evenly_spaced_indices(int(g["h_grid_km"].numel()),  HEIGHT_MAX_COUNT)

        pcfg = replace(bcfg, oscillation=replace(osc, antinu=infer_antinu(ptcl)))

        for i_ang, ai in enumerate(ang_idx.tolist()):
            sel       = select_production_flux(flux_data, ptcl, angle_index=ai, context=ctx)
            theta_deg = float(sel["theta_deg"])
            alpha_deg = sel.get("alpha_deg", None)
            if alpha_deg is not None:
                alpha_deg = float(alpha_deg)

            out_path = build_detector_flux_path(
                str(output_run_dir), ptcl,
                alpha_deg=alpha_deg, theta_deg=theta_deg,
                base_filename=OUTPUT_FILENAME,
            )
            if SKIP_EXISTING and Path(out_path).exists() and not OVERWRITE:
                if DEBUG:
                    print(f"  skip {ptcl} theta={theta_deg:.1f} deg")
                saved.append(out_path)
                continue

            if DEBUG:
                print(
                    f"  {ptcl} [{i_ang+1}/{ang_idx.numel()}]"
                    f" theta={theta_deg:.1f} deg  antinu={infer_antinu(ptcl)}"
                )

            prop  = propagate_particle_angle(
                selected=sel, config=pcfg, profile_earth=ep,
                energy_indices=e_idx, height_indices=h_idx,
            )
            h_grid = as_tensor(sel["h_grid_km"][h_idx], device=ctx.device, dtype=ctx.dtype)

            result = {
                "particle":                ptcl,
                "antinu":                  torch.as_tensor(infer_antinu(ptcl)),
                "alpha_deg":               None if alpha_deg is None else torch.as_tensor(alpha_deg),
                "theta_deg":               torch.as_tensor(theta_deg),
                "E_grid_GeV":              prop["E_grid_GeV"],
                "h_grid_km":               h_grid,
                "detector_flux_Ei":        prop["detector_flux_Ei"],
                "surface_flux_Ei":         prop["surface_flux_Ei"],
                "initial_flux_Ei":         prop["initial_flux_Ei"],
                "detector_probability_Ei": prop["detector_probability_Ei"],
                "surface_probability_Ei":  prop["surface_probability_Ei"],
                "source_flux_E":           prop["source_flux_E"],
                "metadata_extra": {
                    "input_dir":            INPUT_DIR,
                    "earth_density_file":   EARTH_DENSITY_FILE,
                    "detector_depth_m":     float(DETECTOR_DEPTH_M),
                    "matter_in_atmosphere": bool(MATTER_IN_ATMOSPHERE),
                    "reunitarize_earth":    bool(REUNITARIZE_EARTH),
                    "atmosphere_steps":     int(ATMOSPHERE_STEPS),
                    "trajectory_steps":     int(TRAJECTORY_STEPS),
                    "energy_chunk_size":    ENERGY_CHUNK_SIZE,
                    "angle_index":          int(ai),
                    "pmns": {
                        "theta12": float(THETA12), "theta13": float(THETA13),
                        "theta23": float(THETA23), "delta":   float(DELTA_CP),
                    },
                    "mass_splittings_ev2": {
                        "DeltamSq21": float(DM21_EV2),
                        "DeltamSq3l": float(DM3L_EV2),
                    },
                },
            }

            sp = save_detector_flux_result(
                result, str(output_run_dir),
                particle=ptcl, alpha_deg=alpha_deg, theta_deg=theta_deg,
                base_filename=OUTPUT_FILENAME,
                dtype=SAVE_DTYPE, overwrite=OVERWRITE,
            )
            saved.append(sp)
            if DEBUG:
                print(f"  saved: {Path(sp).name}")

    elapsed = time.perf_counter() - t0
    print(f"\nDone. Files: {len(saved)}  Elapsed: {elapsed:.1f} s")
    return saved

In [ ]:
RUN_DIR = next_run_dir(DETECTOR_BASE_DIR)
print(f"Run directory: {RUN_DIR}")

SAVED_PATHS = run_propagation(RUN_DIR)
SAVED_PATHS

## 6. Detector Flux Analysis

Loads the saved detector-flux files from the run directory, aggregates them by
neutrino/antineutrino mode via `aggregate_detector_flux_by_mode`, and produces:

1. **Detector flux spectra by final flavour** — $\Phi_{\rm det}(E,\theta)$ for
   each output flavour and several zenith angles.
2. **Angular profile at reference energy** — $\Phi_{\rm det}(E_{\rm ref},\theta)$
   for each flavour.
3. **Oscillation probability heatmap** — $P_{\rm det}(\nu_\mu \to \nu_\mu; E,\theta)$
   across the full $(E,\theta)$ grid.

**Expected results:** The neutrino mode shows strong $\nu_\mu$ disappearance and
$\nu_\tau$ appearance at $\sim 1$–$10$ GeV for up-going trajectories (large $\theta$).
Antineutrinos follow the same pattern but with slightly different oscillation depth
due to the reversed matter potential ($V \to -V$).

In [ ]:
loaded_data = load_detector_flux_directory(
    str(RUN_DIR),
    map_location="cpu",
    dtype=torch.float64,
    group_by="particle",
)
aggregated = aggregate_detector_flux_by_mode(loaded_data)

print(f"Loaded {sum(len(v) for v in loaded_data.values())} files from {RUN_DIR}")
for mode, agg in aggregated.items():
    n_ang = int(agg["angle_grid_deg"].numel())
    n_E   = int(agg["E_grid_GeV"].numel())
    th_lo = float(agg["theta_grid_deg"].min())
    th_hi = float(agg["theta_grid_deg"].max())
    print(
        f"  Mode '{mode}': {n_ang} angles "
        f"(theta=[{th_lo:.1f},{th_hi:.1f}] deg), {n_E} energies"
    )
    print(f"    Contributors: {agg['particles']}")

### 6.1 Detector Flux Spectra by Final Flavour

$E^2$-weighted detector flux $E^2\,\Phi_{\rm det}(E,\theta)$ for each final flavour
and several zenith angles, shown separately for the neutrino and antineutrino modes.
The $E^2$ weighting flattens the steep power-law and reveals the oscillation-driven
spectral features.

**Expected results:**
- The $\nu_\mu$ spectrum shows a characteristic suppression dip at $\sim 1$–$10$ GeV
  for large $\theta$ (up-going), absent for small $\theta$ (down-going).
- $\nu_\tau$ appearance emerges at $\gtrsim 1$ GeV for large zenith angles where
  $\Delta m^2_{3l}\,L/(4E) \sim \pi/2$.
- The $\nu_e$ channel is nearly unchanged at these energies.

In [ ]:
FLAVOUR_LABELS = [r"$\nu_e$", r"$\nu_\mu$", r"$\nu_\tau$"]
FLAVOUR_COLORS = ["#dd8452", "#4c72b0", "#55a868"]

for mode in aggregated:
    agg     = aggregated[mode]
    E_grid  = to_numpy(agg["E_grid_GeV"])
    theta_g = to_numpy(agg["theta_grid_deg"])
    det_fl  = to_numpy(agg["detector_flux_alpha_Ei"])  # (n_ang, n_E, 3)

    n_ang   = len(theta_g)
    sel_ang = np.linspace(0, n_ang - 1, min(n_ang, 5), dtype=int)
    alphas  = np.linspace(0.3, 1.0, len(sel_ang))
    mode_label = "neutrino" if mode == "nu" else "antineutrino"

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=False)
    for fi, (flabel, fcol) in enumerate(zip(FLAVOUR_LABELS, FLAVOUR_COLORS)):
        ax = axes[fi]
        for ia, alp in zip(sel_ang, alphas):
            y = det_fl[ia, :, fi] * E_grid**2
            lbl = rf"$\theta={theta_g[ia]:.0f}^\circ$" if fi == 0 else None
            ax.loglog(E_grid, y, color=fcol, lw=1.8, alpha=alp, label=lbl)
        ax.set_xlabel("$E$ [GeV]")
        ax.set_ylabel(r"$E^2\,\Phi_{\rm det}$" if fi == 0 else "")
        ax.set_title(f"Final {flabel}", fontsize=10)
        ax.grid(True, which="both", alpha=0.3)
        if fi == 0:
            ax.legend(fontsize=7, ncol=2)

    fig.suptitle(rf"Detector flux — {mode_label} mode, $E^2$-weighted", fontsize=11)
    fig.tight_layout()
    save_and_show(
        f"atm6_fig61_flux_{mode}.png", fig,
        output_dir=OUTPUT_DIR, show_plots=config.show_plots,
    )

### 6.2 Angular Profile at Reference Energy

The detector flux for each final flavour as a function of zenith angle $\theta$ at
a fixed reference energy $E_{\rm ref}$. With $E$ fixed, the angle sets the
baseline $L$ through the Earth, making the $L/E$ dependence explicit.

**Expected results:** A clear dip in the $\nu_\mu$ channel at large $\theta$
(up-going), corresponding to the atmospheric oscillation length at the reference
energy. For $E_{\rm ref} \sim 3$ GeV the first minimum lies near $\theta \approx
150°$–$180°$ (baseline $L \approx 6{,}000$–$12{,}700$ km).

In [ ]:
E_REF_GEV = 3.0

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for col_idx, mode in enumerate(aggregated):
    agg       = aggregated[mode]
    E_grid    = to_numpy(agg["E_grid_GeV"])
    theta_arr = to_numpy(agg["theta_grid_deg"])
    det_fl    = to_numpy(agg["detector_flux_alpha_Ei"])   # (n_ang, n_E, 3)

    ie_ref     = int(np.argmin(np.abs(E_grid - E_REF_GEV)))
    E_actual   = E_grid[ie_ref]
    mode_label = "neutrino" if mode == "nu" else "antineutrino"

    ax = axes[col_idx]
    for fi, (flabel, fcol) in enumerate(zip(FLAVOUR_LABELS, FLAVOUR_COLORS)):
        ax.semilogy(
            theta_arr, det_fl[:, ie_ref, fi],
            "o-", color=fcol, lw=1.8, ms=5,
            label=f"Final {flabel}",
        )
    ax.set_xlabel(r"Zenith angle $\theta$ [deg]")
    ax.set_ylabel(r"$\Phi_{\rm det}(E_{\rm ref},\theta)$")
    ax.set_title(rf"{mode_label} mode, $E_{{\rm ref}} = {E_actual:.2g}$ GeV", fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(True, which="both", alpha=0.3)

fig.suptitle("Angular profile of detector flux at reference energy", fontsize=11)
fig.tight_layout()
save_and_show("atm6_fig62_angular_flux.png", fig, output_dir=OUTPUT_DIR, show_plots=config.show_plots)

### 6.3 Oscillation Probability Heatmap — $\nu_\mu$ Survival

The height-averaged $\nu_\mu$ survival probability
$P_{\rm det}(\nu_\mu \to \nu_\mu; E, \theta) = \Phi_{\rm det}^{\nu_\mu \to \nu_\mu}(E,\theta)
/ \Phi_{\rm src}(E,\theta)$
over the full $(E,\theta)$ grid. The characteristic $L/E$ pattern of atmospheric
neutrino oscillations appears as curved bands across this plane.

**Expected results:** The first disappearance minimum for $\theta = 180°$ is at
$E \approx \Delta m^2_{3l}\,L/(4 \times 1.27) \approx 24$ GeV for $L = 12{,}700$ km.
At $E \sim 3$ GeV the minimum lies near $\theta \approx 150°$. For down-going
directions ($\theta < 90°$) the short baselines preclude significant oscillation
and the probability stays close to 1.

In [ ]:
if "nu" in aggregated:
    agg_nu  = aggregated["nu"]
    E_grid  = to_numpy(agg_nu["E_grid_GeV"])
    theta_g = to_numpy(agg_nu["theta_grid_deg"])
    det_fl  = to_numpy(agg_nu["detector_flux_alpha_Ei"])   # (n_ang, n_E, 3)
    src_fl  = to_numpy(agg_nu["source_flux_alpha_E"])      # (n_ang, n_E)

    FI_NUMU = 1
    eps     = 1e-40
    P_surv  = np.where(src_fl > eps, det_fl[..., FI_NUMU] / src_fl, np.nan)  # (n_ang, n_E)

    fig, ax = plt.subplots(figsize=(10, 5.5))
    im = ax.pcolormesh(
        theta_g, E_grid, P_surv.T,
        cmap="RdYlBu", shading="auto", vmin=0, vmax=1,
    )
    ax.set_yscale("log")
    ax.set_xlabel(r"Zenith angle $\theta$ [deg]")
    ax.set_ylabel("Energy $E$ [GeV]")
    ax.set_title(
        r"$\nu_\mu$ Atmospheric Neutrino Survival Probability at the Detector (Atm. + Earth) $P(\nu_\mu \to \nu_\mu;\ E,\theta)$"
        " — height-averaged",
        fontsize=10,
    )
    fig.colorbar(im, ax=ax, label=r"$P_{\rm det}(\nu_\mu \to \nu_\mu)$")
    ax.grid(True, which="both", alpha=0.2)
    fig.tight_layout()
    save_and_show(
        "atm6_fig63_prob_heatmap.png", fig,
        output_dir=OUTPUT_DIR, show_plots=config.show_plots,
    )
else:
    print("No neutrino-mode data available in aggregated results.")

## 7. Summary

| Step | Function | Output shape |
|------|----------|-------------|
| Load flux | `load_directory(INPUT_DIR, group_by="particle")` | `phi_E_theta_h`: `(n_ang, n_E, n_h)` |
| Select angle | `select_production_flux(flux_data, particle, angle_index=i)` | `selected` dict |
| Atmosphere | `propagate_atmosphere_to_surface(selected, config)` | `surface_probabilities`: `(n_E, n_h, 3)` |
| Earth | `propagate_surface_to_detector(atm, config, profile_earth=ep)` | `detector_probabilities`: `(n_E, n_h, 3)` |
| Height integrate | `integrate_detector_flux_over_height(h, detector_flux_from_production(phi, earth.detector_probabilities))` | `detector_flux_total_Ei`: `(n_E, 3)` |
| Save | `save_detector_flux_result(result, run_dir, ...)` | one `.pt` per (particle, angle) |
| Aggregate | `aggregate_detector_flux_by_mode(loaded_data)` | `detector_flux_alpha_Ei`: `(n_ang, n_E, 3)` |

**Key physical results:**

1. **Atmosphere crossing**: For GeV-scale neutrinos the atmospheric matter effect
   ($n_e^{\rm atm} \sim 10^{-3}\,n_e^{\rm Earth}$) is negligible — surface
   probabilities differ from production probabilities by $\lesssim 10^{-3}$.

2. **Earth matter effect**: Up-going trajectories cross up to 12 700 km of matter.
   The resulting $\nu_\mu$ disappearance is the atmospheric oscillation signature
   discovered by Super-Kamiokande (1998). At $\sim 1$–$10$ GeV and $\theta \approx
   180°$, the survival probability can drop to $\sim 0.1$–$0.3$.

3. **Height integration**: Combining production heights incoherently smooths the
   effective probability $P_{\rm det}(E,\theta)$: each height contributes a different
   $L/E$ ratio, and the average washes out rapid oscillation structure while
   preserving the dominant disappearance signature.

4. **$\nu$ vs $\bar\nu$**: The matter potential reversal ($V \to -V$) produces
   slightly different oscillation depths for neutrinos and antineutrinos; the effect
   is small at GeV energies for $\Delta m^2_{3l}$ but is correctly accounted for by
   the `antinu` flag.

**References**
- Fukuda, Y. et al. (Super-Kamiokande Collaboration, 1998).
  *Evidence for an anomalous disappearance of atmospheric $\nu_\mu$*.
  Phys. Rev. Lett. **81**, 1562.
- Barger, V., Marfatia, D. & Whisnant, K. (2012). *The Physics of Neutrinos*,
  Chapter 8. Princeton University Press.
- Gonzalez-Garcia, M. C. & Nir, Y. (2003).
  *Neutrino masses and mixing: evidence and implications*.
  Rev. Mod. Phys. **75**, 345.

In [ ]:
print("=" * 65)
print("Atmosphere 6 — Detector Flux Generation: Summary")
print("=" * 65)
print(f"  Input flux dir    : {INPUT_DIR}")
print(f"  Output run dir    : {RUN_DIR}")
print(f"  Particles         : {particles}")
print(f"  Files saved       : {len(SAVED_PATHS)}")
print()
for mode in aggregated:
    agg   = aggregated[mode]
    n_ang = int(agg["angle_grid_deg"].numel())
    n_E   = int(agg["E_grid_GeV"].numel())
    print(f"  Mode '{mode}': {n_ang} angles, {n_E} energies, contributors: {agg['particles']}")
print()
print(f"  Oscillation preset: {_osc.preset_name} ({_osc.ordering})")
print(f"  Detector depth    : {DETECTOR_DEPTH_M:.0f} m")
print(f"  Matter in atm     : {MATTER_IN_ATMOSPHERE}")
print(f"  Compute dtype     : {COMPUTE_DTYPE}")
print(f"  Save dtype        : {SAVE_DTYPE}")
print(f"  Output figures    : {OUTPUT_DIR}")